In [32]:
# ==================== STRATEGY SELECTION ====================


STRATEGY_MODE = 2  # 1 = MA-based TP, 2 = S/R-based TP

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pybit.unified_trading import HTTP
from typing import Dict, List, Tuple
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
import random
warnings.filterwarnings('ignore')

In [29]:
# ==================== CONFIGURATION ====================
BYBIT_API_KEY = "br9ADxbhEBOtwG86gZ"
BYBIT_API_SECRET = "YKcVNLqhf6Q5yyqaBXpBmX5bJiNpXETU8FpU"

# PARAMETERS
PARAM_RANGES = {
    'correlation_threshold': (0.1, 1, 0.33),
    'distance_pct': (0.01, 1.01, 0.03),
    'stop_loss_mult': (0.998, 0.948, -0.0015),
    'take_profit_mult': (0.998, 0.948, -0.0015),
    'ma_pairs': [(9, 21), (50, 100), (100, 200)],
    'return_multiplier': (1.5, 7.2, 0.08),
    'volume_mult': (2.0, 12.0, 0.3),
    'entry_mult': (0.998, 0.697, -0.009)
}

CORRELATION_WINDOW = 100
LOOKBACK_PERIOD = 200
POSITION_SIZE_PCT = 0.2

# BOT SETTINGS
MAX_POSITIONS = 400
CHECK_INTERVAL_SECONDS = 120
TOP_N = 96
TIMEFRAME = '15' # MINUTES

# BACKTEST SETTINGS
START_DATE = "2024-12-30 00:00:00"
END_DATE = None
INITIAL_CAPITAL = 25000.0  
LEVERAGE = 1

# ==================== BYBIT CLIENT ====================
session = HTTP(
    testnet=False,
    api_key=BYBIT_API_KEY,
    api_secret=BYBIT_API_SECRET
)

In [34]:
# ==================== DATA STRUCTURES ====================
class BacktestPosition:
    def __init__(self, symbol, direction, entry_price, entry_time, stop_loss, 
                 quantity, ma_short, ma_long, correlation, sr_level, equity_at_entry):
        self.symbol = symbol
        self.direction = direction
        self.entry_price = entry_price
        self.entry_time = entry_time
        self.stop_loss = stop_loss
        self.trailing_stop = stop_loss
        self.tp_target_set = False
        self.quantity = quantity
        self.ma_short = ma_short
        self.ma_long = ma_long
        self.correlation = correlation
        self.sr_level = sr_level
        self.equity_at_entry = equity_at_entry
        self.exit_price = None
        self.exit_time = None
        self.exit_reason = None
        self.pnl = 0.0
        self.pnl_pct = 0.0

class BacktestEngine:
    def __init__(self, initial_capital):
        self.initial_capital = initial_capital
        self.equity = initial_capital
        self.positions = {}
        self.closed_trades = []
        self.equity_curve = []
        self.max_equity = initial_capital
        self.max_drawdown = 0.0
        
    def calculate_position_size(self, symbol, entry_price: float, stop_loss: float) -> float:
        info = get_symbol_info(symbol)
        if not info:
            print(f'{symbol}: Skipping order due to missing instument info.')
            return None

        qty_step = info['qtyStep']
        tick_size = info['tickSize']
        
        risk_amount = self.equity * (POSITION_SIZE_PCT / 100)
        risk_per_unit = abs(entry_price - stop_loss) / entry_price
        
        if risk_per_unit == 0:
            return 0.0

        qty = (risk_amount / risk_per_unit) / entry_price
        qty = round(qty, 3)
        rounded_qty = (qty // qty_step) * qty_step
        qty_decimals = len(str(qty_step).split('.')[-1]) if '.' in str(qty_step) else 0
        rounded_qty = abs(round(rounded_qty, qty_decimals))

        if rounded_qty <= 0:
            print(f'{symbol}: Rounded quantity is <= 0 ({rounded_qty}). Skipping')
            return None
        
        return float(rounded_qty)
    
    def open_position(self, symbol, signal, current_time):
        if len(self.positions) >= MAX_POSITIONS:
            return False
        
        if symbol in self.positions:
            return False
        
        qty = self.calculate_position_size(symbol, signal['entry_price'], signal['stop_loss'])
        
        if qty is None:
            return False
        
        position = BacktestPosition(
            symbol=symbol,
            direction=signal['direction'],
            entry_price=signal['entry_price'],
            entry_time=current_time,
            stop_loss=signal['stop_loss'],
            quantity=qty,
            ma_short=signal['ma_short'],
            ma_long=signal['ma_long'],
            correlation=signal['correlation'],
            sr_level=signal['sr_level'],
            equity_at_entry=self.equity
        )
        
        self.positions[symbol] = position
        return True
    
    def close_position(self, symbol, exit_price, exit_time, reason):
        if symbol not in self.positions:
            return
        
        position = self.positions[symbol]
        position.exit_price = exit_price
        position.exit_time = exit_time
        position.exit_reason = reason
        
        if position.direction == 'long':
            price_change = exit_price - position.entry_price
        else:
            price_change = position.entry_price - exit_price
        
        position.pnl = (price_change / position.entry_price) * position.quantity * position.entry_price * LEVERAGE
        position.pnl_pct = (position.pnl / position.equity_at_entry) * 100
        
        self.equity += position.pnl
        self.closed_trades.append(position)
        
        if self.equity > self.max_equity:
            self.max_equity = self.equity
        
        drawdown = (self.max_equity - self.equity) / self.max_equity * 100
        if drawdown > self.max_drawdown:
            self.max_drawdown = drawdown
        
        del self.positions[symbol]
    
    def check_stop_loss(self, symbol, current_low, current_high, current_time):
        if symbol not in self.positions:
            return
        
        position = self.positions[symbol]
        
        if position.direction == 'long':
            if current_low <= position.trailing_stop:
                self.close_position(symbol, position.trailing_stop, current_time, 'stop_loss')
        else:
            if current_high >= position.trailing_stop:
                self.close_position(symbol, position.trailing_stop, current_time, 'stop_loss')
    
    def check_take_profit(self, symbol, current_price, ma_short, ma_long, current_time, config, sr_levels=None):
        if symbol not in self.positions:
            return
        
        position = self.positions[symbol]
        
        if pd.isna(ma_short) or pd.isna(ma_long):
            return
        
        if position.direction == 'long':
            if STRATEGY_MODE == 1:
                tp_short = ma_short * config['take_profit_mult']
                tp_long = ma_long * config['take_profit_mult']
                tp_price = min(tp_short, tp_long)

                if tp_price > position.entry_price * (2 - config['entry_mult']):
                    position.trailing_stop = max(position.trailing_stop, tp_price)



            elif STRATEGY_MODE == 2:
                if not position.tp_target_set and sr_levels and sr_levels['resistance']:
                    nearest_resistance = sr_levels['resistance'][0]

                    if (nearest_resistance > position.entry_price * (2 - config['entry_mult']) and current_price >= nearest_resistance):
                        position.trailing_stop = nearest_resistance
                        position.tp_target_set = True

        
        else: 
            if STRATEGY_MODE == 1:
                tp_short = ma_short * (2 - config['take_profit_mult'])
                tp_long = ma_long * (2 - config['take_profit_mult'])
                tp_price = max(tp_short, tp_long)
                
                if tp_price < position.entry_price * config['entry_mult']:
                    position.trailing_stop = min(position.trailing_stop, tp_price)
                        

            elif STRATEGY_MODE == 2:
                if not position.tp_target_set and sr_levels and sr_levels['support']:
                    nearest_support = sr_levels['support'][0]
                    
                    if (nearest_support < position.entry_price * config['entry_mult'] and current_price <= nearest_support):
                        position.trailing_stop = nearest_support
                        position.tp_target_set = True 
                    
    
    def record_equity(self, timestamp):
        self.equity_curve.append({
            'timestamp': timestamp,
            'equity': self.equity,
            'open_positions': len(self.positions)
        })

In [7]:
# ==================== CONFIG GENERATOR ====================
def generate_configs(num_systematic=30, num_random=100):
    configs = []
    corr_range = np.arange(PARAM_RANGES['correlation_threshold'][0], PARAM_RANGES['correlation_threshold'][1], PARAM_RANGES['correlation_threshold'][2])[:30]
    vol_range = np.arange(PARAM_RANGES['volume_mult'][0], PARAM_RANGES['volume_mult'][1], PARAM_RANGES['volume_mult'][2])[:30]
    dist_range = np.arange(PARAM_RANGES['distance_pct'][0], PARAM_RANGES['distance_pct'][1], PARAM_RANGES['distance_pct'][2])[:30]
    sl_range = np.arange(PARAM_RANGES['stop_loss_mult'][0], PARAM_RANGES['stop_loss_mult'][1], PARAM_RANGES['stop_loss_mult'][2])[:30]
    tp_range = np.arange(PARAM_RANGES['take_profit_mult'][0], PARAM_RANGES['take_profit_mult'][1], PARAM_RANGES['take_profit_mult'][2])[:30]
    return_mult = np.arange(PARAM_RANGES['return_multiplier'][0], PARAM_RANGES['return_multiplier'][1], PARAM_RANGES['return_multiplier'][2])[:30]
    entry_mult = np.arange(PARAM_RANGES['entry_mult'][0], PARAM_RANGES['entry_mult'][1], PARAM_RANGES['entry_mult'][2])[:30]
    
    base_30 = []
    for i in range(30):
        base_30.append({
            'correlation_threshold': corr_range[i] if i < len(corr_range) else corr_range[-1],
            'volume_mult': vol_range[i] if i < len(vol_range) else vol_range[-1],
            'distance_pct': dist_range[i] if i < len(dist_range) else dist_range[-1],
            'stop_loss_mult': sl_range[i] if i < len(sl_range) else sl_range[-1],
            'take_profit_mult': tp_range[i] if i < len(tp_range) else tp_range[-1],
            'return_multiplier': return_mult[i] if i < len(return_mult) else return_mult[-1],
            'entry_mult': entry_mult[i] if i < len(entry_mult) else entry_mult[-1],
            'ma_pair': PARAM_RANGES['ma_pairs'][0]
        })
    
    configs.extend(base_30)
    
    for ma_pair in PARAM_RANGES['ma_pairs'][1:]:
        for config in base_30:
            new_config = config.copy()
            new_config['ma_pair'] = ma_pair
            configs.append(new_config)
    
    for _ in range(num_random):
        configs.append({
            'correlation_threshold': random.uniform(*PARAM_RANGES['correlation_threshold'][:2]),
            'volume_mult': random.uniform(*PARAM_RANGES['volume_mult'][:2]),
            'distance_pct': random.uniform(*PARAM_RANGES['distance_pct'][:2]),
            'stop_loss_mult': random.uniform(PARAM_RANGES['stop_loss_mult'][1], PARAM_RANGES['stop_loss_mult'][0]),
            'take_profit_mult': random.uniform(PARAM_RANGES['take_profit_mult'][1], PARAM_RANGES['take_profit_mult'][0]),
            'return_multiplier': random.uniform(*PARAM_RANGES['return_multiplier'][:2]),
            'entry_mult': random.uniform(PARAM_RANGES['entry_mult'][1], PARAM_RANGES['entry_mult'][0]),
            'ma_pair': random.choice(PARAM_RANGES['ma_pairs'])
        })
    
    return configs
    

In [16]:
# ==================== TECHNICAL FUNCTIONS ====================
def calculate_pivots(H: float, L: float, C: float) -> Dict:
    levels = {}
    P = (H + L + C) / 3.0
    levels['standard'] = {'P': P, 'R1': 2*P-L, 'S1': 2*P-H, 'R2': P+(H-L), 'S2': P-(H-L), 'R3': H+2*(P-L), 'S3': L-2*(H-P)}
    rng = H - L
    levels['fibonacci'] = {'P': P, 'R1': P+0.382*rng, 'S1': P-0.382*rng, 'R2': P+0.618*rng, 'S2': P-0.618*rng, 'R3': P+rng, 'S3': P-rng}
    WP = (H + L + 2*C) / 4.0
    levels['woodie'] = {'P': WP, 'R1': 2*WP-L, 'S1': 2*WP-H, 'R2': WP+(H-L), 'S2': WP-(H-L)}
    cpr_P = (H + L + C) / 3.0
    cpr_BC = (H + L) / 2.0
    cpr_TC = 2*cpr_P - cpr_BC
    lo, hi = sorted([cpr_BC, cpr_TC])
    levels['cpr'] = {'P': cpr_P, 'BC': lo, 'TC': hi}
    cR1, cS1 = C+(H-L)*1.1/12, C-(H-L)*1.1/12
    cR2, cS2 = C+(H-L)*1.1/6, C-(H-L)*1.1/6
    cR3, cS3 = C+(H-L)*1.1/4, C-(H-L)*1.1/4
    cR4, cS4 = C+(H-L)*1.1/2, C-(H-L)*1.1/2
    levels['camarilla'] = {'R1':cR1,'S1':cS1,'R2':cR2,'S2':cS2,'R3':cR3,'S3':cS3,'R4':cR4,'S4':cS4}
    if C < H+L-C: X = H+2*L+C
    elif C > H+L-C: X = 2*H+L+C
    else: X = H+L+2*C
    levels['demark'] = {'P': X/4, 'R1': X/2-L, 'S1': X/2-H}
    return levels

def calculate_poc(df):
    df = df.copy()
    df['typical_price'] = (df['high'] + df['low'] + df['close']) / 3
    price_volume = df.groupby('typical_price')['volume_usdt'].sum()
    poc = price_volume.idxmax()
    return poc

def get_support_resistance(df, volume_mult) -> Dict:
    levels = {'support': [], 'resistance': []}
    H, L, C = df['high'].max(), df['low'].min(), df['close'].iloc[-1]
    current_price = C
    piv = calculate_pivots(H, L, C)
    for style_name, style_levels in piv.items():
        for k, v in style_levels.items():
            if v and v < current_price:
                levels['support'].append(v)
            elif v and v > current_price:
                levels['resistance'].append(v)
    try:
        poc = calculate_poc(df)
        if poc < current_price: levels['support'].append(poc)
        elif poc > current_price: levels['resistance'].append(poc)
    except:
        pass
    volume_ma = df['volume_usdt'].rolling(window=20).mean()
    
    for i in range(20, len(df) - 20):
        if (df['low'].iloc[i] < df['low'].iloc[i-10:i].min() and 
            df['low'].iloc[i] < df['low'].iloc[i+1:i+11].min() and
            df['volume_usdt'].iloc[i] > volume_ma.iloc[i] * volume_mult):
            swing_low = df['low'].iloc[i]
            if swing_low < current_price:
                levels['support'].append(swing_low)
    
    for i in range(20, len(df) - 20):
        if (df['high'].iloc[i] > df['high'].iloc[i-10:i].max() and 
            df['high'].iloc[i] > df['high'].iloc[i+1:i+11].max() and
            df['volume_usdt'].iloc[i] > volume_ma.iloc[i] * volume_mult):
            swing_high = df['high'].iloc[i]
            if swing_high > current_price:
                levels['resistance'].append(swing_high)
    
    levels['support'] = sorted(list(set(levels['support'])), reverse=True)
    levels['resistance'] = sorted(list(set(levels['resistance'])))
    levels['current_price'] = current_price
    
    return levels

def calculate_mas(df: pd.DataFrame, short: int = 100, long: int = 200) -> pd.DataFrame:
    df = df.copy()
    df[f'ma_{short}'] = df['close'].rolling(window=short).mean()
    df[f'ma_{long}'] = df['close'].rolling(window=long).mean()
    return df


def fetch_klines(symbol: str, interval: str, limit: int = 1000, total_candles: int = 5000) -> pd.DataFrame:
    try:
        all_klines = []
        fetched = 0
        end_time = END_DATE
        
        calls_needed = (total_candles + limit - 1) // limit
        for call in range(calls_needed):
            candles_to_fetch = min(limit, total_candles - fetched)
            
            params = {
                'category': 'linear',
                'symbol': symbol,
                'interval': interval,
                'limit': candles_to_fetch
            }
            
            if end_time is not None:
                params['end'] = end_time
            
            response = session.get_kline(**params)
            if response['retCode'] != 0:
                print(f'Error fetching {symbol}: {response.get('retMsg', 'Unknown error')}')
                break
            klines = response['result']['list']
            if not klines:
                break 
            all_klines.extend(klines)
            fetched += len(klines)
            
            oldest_timestamp = int(klines[-1][0])
            end_time = oldest_timestamp - 1
            if fetched >= total_candles:
                break
            
            time.sleep(0.1)
        
        if not all_klines:
            return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume', 'turnover'])
        df['timestamp'] = pd.to_datetime(df['timestamp'].astype(float), unit='ms')
        for col in ['open', 'high', 'low', 'close', 'volume']:
            df[col] = df[col].astype(float)
        df = df.sort_values('timestamp').drop_duplicates(subset=['timestamp']).reset_index(drop=True)
        df['returns'] = df['close'].pct_change() * 100
        df['volume_usdt'] = df['volume'] * df['close']
        
        return df[['timestamp', 'open', 'high', 'low', 'close', 'volume', 'volume_usdt', 'returns']]
        
    except Exception as e:
        print(f"Error fetching {symbol}: {e}")
        time.sleep(1)
        return pd.DataFrame()

def get_top_symbols() -> List[str]:
    try:
        response = session.get_tickers(category="linear")
        if response['retCode'] != 0:
            return []
        
        tickers = response['result']['list']
        df_tickers = pd.DataFrame(tickers)
        
        df_tickers = df_tickers[df_tickers['symbol'].str.endswith('USDT')].copy()
        df_tickers = df_tickers[df_tickers['symbol'] != 'BTCUSDT']
        
        df_tickers['turnover24h'] = df_tickers['turnover24h'].astype(float)
        df_tickers = df_tickers.sort_values('turnover24h', ascending=False)
        
        top_symbols = df_tickers.head(TOP_N + 10)['symbol'].tolist()
        top_symbols = [s for s in top_symbols if s != 'BTCUSDT' and s != 'DASHUSDT' and s != 'BATUSDT' and s != 'AI16ZUSDT' and s != 'WLFIUSDT'][:TOP_N]
        
        return top_symbols
    except:
        return []

SYMBOL_INFO_CACHE = {}

def get_symbol_info(symbol: str) -> dict:
    global SYMBOL_INFO_CACHE
    
    if symbol in SYMBOL_INFO_CACHE:
        return SYMBOL_INFO_CACHE[symbol]
    
    try:
        response = session.get_instruments_info(
            category='linear',
            symbol=symbol
        )
        
        if response['retCode'] == 0 and response['result']['list']:
            info = response['result']['list'][0]
            qty_step = float(info['lotSizeFilter']['qtyStep'])
            tick_size = float(info['priceFilter']['tickSize'])
            
            SYMBOL_INFO_CACHE[symbol] = {
                'qtyStep': qty_step,
                'tickSize': tick_size
            }
            return SYMBOL_INFO_CACHE[symbol]
        else:
            print(f"Failed to fetch instrument info for {symbol}: {response.get('retMsg', 'Unknown error')}")
            return None
            
    except Exception as e:
        print(f"Exception fetching instrument info for {symbol}: {e}")
        return None
            
    

def check_strategy1_signal(btc_df: pd.DataFrame, coin_df: pd.DataFrame, symbol: str, config: Dict):
    try:
        if len(btc_df) < CORRELATION_WINDOW + LOOKBACK_PERIOD + 10:
            return None
        if len(coin_df) < CORRELATION_WINDOW + LOOKBACK_PERIOD + 10:
            return None
            
        coin_df = calculate_mas(coin_df, config['ma_pair'][0], config['ma_pair'][1])
        btc_returns = btc_df['returns']
        coin_returns = coin_df['returns']
        aligned_btc, aligned_coin = btc_returns.align(coin_returns, join='inner')

        if len(aligned_btc) < CORRELATION_WINDOW + 10:
            return None
        
        i = len(aligned_btc) - 1
        corr_window = aligned_btc.iloc[i-CORRELATION_WINDOW:i].corr(aligned_coin.iloc[i-CORRELATION_WINDOW:i])
        
        if pd.isna(corr_window) or corr_window < config['correlation_threshold']:
            return None
        
        btc_return = aligned_btc.iloc[i]
        coin_return = aligned_coin.iloc[i]
        
        if pd.isna(btc_return) or pd.isna(coin_return):
            return None
        
        if abs(coin_return) < abs(btc_return) * config['return_multiplier']:
            return None
        
        btc_idx = len(btc_df) - 1
        if btc_idx < LOOKBACK_PERIOD:
            return None
            
        coin_idx = len(coin_df) - 1
        if coin_idx < LOOKBACK_PERIOD:
            return None
        
        lookback_df = btc_df.iloc[btc_idx-LOOKBACK_PERIOD:btc_idx]
        if len(lookback_df) < 50:
            return None
        btc_current_low = btc_df['low'].iloc[btc_idx]   
        btc_current_high = btc_df['high'].iloc[btc_idx]
        coin_current_price = coin_df['close'].iloc[coin_idx]
        ma_short = coin_df[f'ma_{config["ma_pair"][0]}'].iloc[coin_idx]
        ma_long = coin_df[f'ma_{config["ma_pair"][1]}'].iloc[coin_idx]
        
        if pd.isna(ma_short) or pd.isna(ma_long):
            return None
        
        levels = get_support_resistance(lookback_df, config['volume_mult'])
        if not levels['support'] or not levels['resistance']:
            return None
        
        nearest_support = levels['support'][0]
        nearest_resistance = levels['resistance'][0]
        distance_to_support = abs(btc_current_low - nearest_support) / nearest_support * 100
        distance_to_resistance = abs(btc_current_high - nearest_resistance) / nearest_resistance * 100

        current_15m_volume = coin_df['volume_usdt'].iloc[-1]

        if distance_to_support <= config['distance_pct']:
            return {
                'direction': 'long',
                'entry_price': coin_current_price,
                'stop_loss': coin_current_price * config['stop_loss_mult'],
                'ma_short': ma_short,
                'ma_long': ma_long,
                'correlation': corr_window,
                'sr_level': nearest_support,
                'volume_15m': current_15m_volume
            }
        elif distance_to_resistance <= config['distance_pct']:
            return {
                'direction': 'short',
                'entry_price': coin_current_price,
                'stop_loss': coin_current_price * (2 - config['stop_loss_mult']),
                'ma_short': ma_short,
                'ma_long': ma_long,
                'correlation': corr_window,
                'sr_level': nearest_resistance,
                'volume_15m': current_15m_volume
            }
        return None
    except:
        return None

In [1]:
# ==================== MAIN ====================
def run_backtest():
    print('='*80)
    print('BACKTEST SIMULATION')
    print('='*80)
    print(f'Start Date: {START_DATE}')
    print(f'Initial Capital: ${INITIAL_CAPITAL:.2f}')
    print(f'Timeframe: {TIMEFRAME}m')
    print(f'Max Positions: {MAX_POSITIONS}')
    print(f'Position Size: {POSITION_SIZE_PCT}% risk per trade')
    print(f'Leverage: {LEVERAGE}x')
    print(f'Strategy Mode: {STRATEGY_MODE} (1=MA-based TP, 2=S/R-based TP)')
    print('='*80)

    engine = BacktestEngine(INITIAL_CAPITAL)

    print('\nFetching historical data...')
    btc_df = fetch_klines("BTCUSDT", TIMEFRAME, limit=1000, total_candles=1000)

    if btc_df.empty:
        print('Failed to fetch BTC data!')
        return
    
    start_dt = pd.to_datetime(START_DATE)
    btc_df = btc_df[btc_df['timestamp'] >= start_dt].reset_index(drop=True)
    
    if len(btc_df) == 0:
        print(f'No data available from {START_DATE}')
        return
    
    print(f'BTC data: {len(btc_df)} candles from {btc_df['timestamp'].iloc[0]} to {btc_df['timestamp'].iloc[-1]}')
    
    symbols = get_top_symbols()
    print(f'Analyzing {len(symbols)} symbols...')

    symbol_data_cache = {}
    for symbol in symbols:
        df = fetch_klines(symbol, TIMEFRAME, limit=1000, total_candles=1000)
        if not df.empty:
            df = df[df['timestamp'] >= start_dt].reset_index(drop=True)
            symbol_data_cache[symbol] = df
        time.sleep(0.1)
    
    print(f'Cached data for {len(symbol_data_cache)} symbols')
    print('\nRunning backtest simulation...')
    print('='*80)
    
    total_candles = len(btc_df)

    configs = generate_configs()
    print(f'\n✅ Generated {len(configs)} configurations')

    engines = {i: BacktestEngine(INITIAL_CAPITAL) for i, _ in enumerate(configs)}

    print('Running simulation...')
    for candle_idx in range(LOOKBACK_PERIOD + CORRELATION_WINDOW + 10, total_candles):
        
        if candle_idx % 100 == 0:
            progress_pct = (candle_idx / total_candles) * 100
            print(f'[{datetime.now().strftime('%H:%M:%S')}] ⏳ Progress: {candle_idx}/{total_candles} ({progress_pct:.2f}%) | Market Time: {current_time}')
        
        current_time = btc_df['timestamp'].iloc[candle_idx]
        btc_slice = btc_df.iloc[:candle_idx+1].copy()
        
        for config_id, config in enumerate(configs):
            current_engine = engines[config_id]

            current_sr_levels = None
            if STRATEGY_MODE == 2:
                lookback_for_sr = btc_df.iloc[max(0, candle_idx-LOOKBACK_PERIOD):candle_idx]
                if len(lookback_for_sr) >= 50:
                    current_sr_levels = get_support_resistance(lookback_for_sr, config['volume_mult'])
     
            
            for symbol in list(current_engine.positions.keys()):
                if symbol not in symbol_data_cache: continue
                
                coin_df = symbol_data_cache[symbol]
                if len(coin_df) <= candle_idx: continue
                
                current_candle = coin_df.iloc[candle_idx]
                
                if current_candle['timestamp'] != current_time:
                    continue 
                    
                current_engine.check_stop_loss(symbol, current_candle['low'], current_candle['high'], current_time)
                
                if symbol in current_engine.positions:
                    coin_slice = coin_df.iloc[candle_idx-205:candle_idx+1] 
                    
                    coin_slice = calculate_mas(coin_slice, config['ma_pair'][0], config['ma_pair'][1])
                    if len(coin_slice) > 0:
                        ma_short = coin_slice[f'ma_{config["ma_pair"][0]}'].iloc[-1]
                        ma_long = coin_slice[f'ma_{config["ma_pair"][1]}'].iloc[-1]
                        current_engine.check_take_profit(symbol, current_candle['close'], ma_short, ma_long, current_time, config, sr_levels=current_sr_levels)

            if len(current_engine.positions) < MAX_POSITIONS:
                
                for symbol in symbols:
                    if symbol in current_engine.positions: continue
                    if symbol not in symbol_data_cache: continue
                    
                    coin_df = symbol_data_cache[symbol]
                    if len(coin_df) <= candle_idx: continue
                    
                    coin_slice = coin_df.iloc[:candle_idx+1]
                    if len(coin_slice) < CORRELATION_WINDOW + LOOKBACK_PERIOD + 10: continue

                    signal = check_strategy1_signal(btc_slice, coin_slice, symbol, config)
                    
                    if signal:
                        current_engine.open_position(symbol, signal, current_time)
                        if len(current_engine.positions) >= MAX_POSITIONS:
                            break
            
            current_engine.record_equity(current_time)

    print("\nCalculating Final Results...")
    
    all_bot_results = []
    
    for config_id, config in enumerate(configs):
        engine = engines[config_id]
        
        for symbol in list(engine.positions.keys()):
             if symbol in symbol_data_cache:
                final_price = symbol_data_cache[symbol].iloc[-1]['close']
                final_time = symbol_data_cache[symbol].iloc[-1]['timestamp']
                engine.close_position(symbol, final_price, final_time, 'backtest_end')

        trades_df = pd.DataFrame([t.__dict__ for t in engine.closed_trades])
        
        total_trades = len(trades_df)
        if total_trades > 0:
            wins = len(trades_df[trades_df['pnl'] > 0])
            win_rate = (wins / total_trades) * 100
            total_return_usd = engine.equity - engine.initial_capital
            total_return_pct = (total_return_usd / engine.initial_capital) * 100
            
            all_bot_results.append({
                'config_id': config_id,
                'config': config,
                'final_equity': engine.equity,
                'total_return_pct': total_return_pct,
                'win_rate': win_rate,
                'total_trades': total_trades,
                'max_drawdown': engine.max_drawdown,
                'trades_df': trades_df
            })

    results_df = pd.DataFrame(all_bot_results)
    df_filtered = results_df[results_df['total_trades'] >= 10].copy()

    print('\n' + '='*80)
    print('TOP 20 BEST RETURNS:')
    print('='*80)
    
    for idx, row in df_filtered.nlargest(20, 'total_return_pct').iterrows():
        symbol_stat = row['trades_df'].groupby('symbol').agg(
            total_trades=('symbol', 'size'), 
            wins=('pnl', lambda x: (x > 0).sum())
        ).reset_index()
        
        symbol_stat['win_rate'] = (symbol_stat['wins'] / symbol_stat['total_trades']) * 100
        symbol_stat_wr = symbol_stat[(symbol_stat['win_rate'] < 20) & (symbol_stat['total_trades'] >= 50)]
        
        print(f'Bot {row['config_id']} | | Return: {row['total_return_pct']:.2f}% | WR: {row['win_rate']:.2f}% | Trades: {row['total_trades']}')
        avg_win = row['trades_df'][row['trades_df']['pnl'] > 0]['pnl_pct'].mean() if row['win_rate'] > 0 else 0
        avg_loss = row['trades_df'][row['trades_df']['pnl'] <= 0]['pnl_pct'].mean() if row['win_rate'] < 100 else 0
        
        print(f'  AvgWin: {avg_win:.2f}% | AvgLoss: {avg_loss:.2f}% | Max Drawdown: {row['max_drawdown']}% | Final Equity: {row['final_equity']}')
        print(f'  Vol={row['config']['volume_mult']:.1f}, Dist={row['config']['distance_pct']:.3f}, Corr={row['config']['correlation_threshold']:.2f}')
        print(f'          SL={row['config']['stop_loss_mult']:.4f}, TP={row['config']['take_profit_mult']:.4f}, MA={row['config']['ma_pair']}, RETURN={row['config']['return_multiplier']:.2f}, ENTRY={row['config']['entry_mult']:.3f}')
        
        for _, r in symbol_stat_wr.iterrows():
            print(f'    WARNING Symbol: {r['symbol']} | Win Rate: {r['win_rate']:.2f}% | Trades: {r['total_trades']}')
        print('-' * 80)

    print('\n' + '='*80)
    print('TOP 20 BEST WIN RATE:')
    print('='*80)
    for idx, row in df_filtered.nlargest(20, 'win_rate').iterrows():
        print(f'Bot {row['config_id']} | WR: {row['win_rate']:.2f}% | Return: {row['total_return_pct']:.2f}% | Trades: {row['total_trades']}')
        avg_win = row['trades_df'][row['trades_df']['pnl'] > 0]['pnl_pct'].mean() if row['win_rate'] > 0 else 0
        avg_loss = row['trades_df'][row['trades_df']['pnl'] <= 0]['pnl_pct'].mean() if row['win_rate'] < 100 else 0
        
        print(f'  AvgWin: {avg_win:.2f}% | AvgLoss: {avg_loss:.2f}% | Max Drawdown: {row['max_drawdown']}% | Final Equity: {row['final_equity']}')
        print(f'  Vol={row['config']['volume_mult']:.1f}, Dist={row['config']['distance_pct']:.3f}, Corr={row['config']['correlation_threshold']:.2f}')
        print(f'          SL={row['config']['stop_loss_mult']:.4f}, TP={row['config']['take_profit_mult']:.4f}, MA={row['config']['ma_pair']}, RETURN={row['config']['return_multiplier']:.2f}, ENTRY={row['config']['entry_mult']:.3f}')
        

    print('\n' + '='*89)
    print('TOP 20 WORST RETURNS:')
    print('='*80)
    for idx, row in df_filtered.nsmallest(20, 'total_return_pct').iterrows():
        print(f'Bot {row['config_id']} | Return: {row['total_return_pct']:.2f}% | WR: {row['win_rate']:.2f}% | Trades: {row['total_trades']}')
        print(f'  Vol={row['config']['volume_mult']:.1f}, Dist={row['config']['distance_pct']:.3f}, Corr={row['config']['correlation_threshold']:.2f}')
        print(f'          SL={row['config']['stop_loss_mult']:.4f}, TP={row['config']['take_profit_mult']:.4f}, MA={row['config']['ma_pair']}, RETURN={row['config']['return_multiplier']:.2f}, ENTRY={row['config']['entry_mult']:.3f}')
        

    # ==================== SYMBOL ANALYSIS ====================
    print(f'\n Top 10 Best & Worst Performing Symbols across ALL configs:')
    print('='*80)

    all_trades_dfs = results_df['trades_df'].tolist()

    if all_trades_dfs:
        df_all_trades = pd.concat(all_trades_dfs, ignore_index=True)
        
        symbol_performance = df_all_trades.groupby('symbol').agg(
            total_trades=('symbol', 'size'),
            avg_return_pct=('pnl_pct', 'mean'),
            wins=('pnl', lambda x: (x > 0).sum()),
            total_pnl=('pnl', 'sum')
        ).reset_index()

        symbol_performance['win_rate'] = (symbol_performance['wins'] / symbol_performance['total_trades']) * 100
        
        df_symbols_filtered = symbol_performance[symbol_performance['total_trades'] >= 50].copy()

        if not df_symbols_filtered.empty:
            print('\n--- BEST PERFORMING (By w/r): ---')
            df_best_symbols = df_symbols_filtered.nlargest(10, 'win_rate')
            for _, row in df_best_symbols.iterrows():
                print(f'Symbol: {row['symbol']:<10} | Total Return: {row['total_pnl']} | Avg Return: {row['avg_return_pct']:.2f}% | Win Rate: {row['win_rate']:.2f}% | Trades: {row['total_trades']}')

            print('\n--- WORST PERFORMING (By w/r): ---')
            df_worst_symbols = df_symbols_filtered.nsmallest(10, 'win_rate')
            for _, row in df_worst_symbols.iterrows():
                print(f'Symbol: {row['symbol']:<10} | Total Return: {row['total_pnl']} | Avg Return: {row['avg_return_pct']:.2f}% | Win Rate: {row['win_rate']:.2f}% | Trades: {row['total_trades']}')

            print('\n--- BEST PERFORMING (By return): ---')
            df1_best_symbols = df_symbols_filtered.nlargest(10, 'total_pnl')
            for _, row in df1_best_symbols.iterrows():
                print(f'Symbol: {row['symbol']:<10} | Total Return: {row['total_pnl']} | Avg Return: {row['avg_return_pct']:.2f}% | Win Rate: {row['win_rate']:.2f}% | Trades: {row['total_trades']}')

            print('\n--- WORST PERFORMING (By return): ---')
            df1_worst_symbols = df_symbols_filtered.nsmallest(10, 'total_pnl')
            for _, row in df1_worst_symbols.iterrows():
                print(f'Symbol: {row['symbol']:<10} | Total Return: {row['total_pnl']} | Avg Return: {row['avg_return_pct']:.2f}% | Win Rate: {row['win_rate']:.2f}% | Trades: {row['total_trades']}')
                

        else:
            print('Not enough trade data per symbol for analysis (min 50 trades total across all configs).')
    else:
        print('No trade data available for symbol analysis.')
    
    return results_df



In [35]:
if __name__ == '__main__':
    engine = run_backtest()

BACKTEST SIMULATION
Start Date: 2024-12-30 00:00:00
Initial Capital: $25000.00
Timeframe: 15m
Max Positions: 400
Position Size: 0.2% risk per trade
Leverage: 1x
Strategy Mode: 2 (1=MA-based TP, 2=S/R-based TP)

Fetching historical data...
BTC data: 1000 candles from 2026-01-03 08:15:00 to 2026-01-13 18:00:00
Analyzing 96 symbols...
Cached data for 96 symbols

Running backtest simulation...

✅ Generated 190 configurations
Running simulation...
[13:47:15] ⏳ Progress: 400/1000 (40.00%) | Market Time: 2026-01-07 12:00:00
[14:25:11] ⏳ Progress: 500/1000 (50.00%) | Market Time: 2026-01-08 13:00:00
[14:59:43] ⏳ Progress: 600/1000 (60.00%) | Market Time: 2026-01-09 14:00:00
[15:35:04] ⏳ Progress: 700/1000 (70.00%) | Market Time: 2026-01-10 15:00:00
[16:06:21] ⏳ Progress: 800/1000 (80.00%) | Market Time: 2026-01-11 16:00:00
[16:40:44] ⏳ Progress: 900/1000 (90.00%) | Market Time: 2026-01-12 17:00:00

Calculating Final Results...

TOP 20 BEST RETURNS:
Bot 1 | | Return: 105.96% | WR: 5.18% | Trade